# Phase 1 Slurm Batch Controller

This notebook reads a loci manifest and writes a Slurm job-array script that runs steps 1 and 2 (z-score export and LD/metrics) with one locus per task.

In [ ]:
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    candidates = []
    current = (start or Path.cwd()).resolve()
    candidates.extend([current, *current.parents])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate

    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.locus_manifest import load_loci_manifest, validate_loci_manifest
from utils.paths import configure_runtime_env

PATHS = configure_runtime_env(PROJECT_ROOT)
MANIFEST_PATH = PROJECT_ROOT / 'output' / 'prelim' / 'phase1_metrics_screening_review' / 'representative_gene_sample_n10_manifest.csv'
JOB_NAME = 'phase1_selected_manifest_10_loci'
RUN_MODE = 'processing'  # 'screening' writes only aggregate dataset metrics; 'processing' saves full phase-1 outputs
LOCI_PER_TASK = 1
ACCOUNT = 'your_slurm_account'  # example -- set to your environment
MAIL_USER = 'you@example.com'
MAIL_TYPE = 'BEGIN,END,FAIL'
TIME_LIMIT = '02:00:00'
MEMORY = '8G'
CPUS_PER_TASK = 1
FORCE_RERUN = True

OUTPUT_DIR = PATHS.output
PRELIM_DIR = PATHS.output_prelim
SLURM_SCRIPT_DIR = OUTPUT_DIR / 'slurm_scripts'
SLURM_PRINTS_DIR = OUTPUT_DIR / 'slurm_prints'
BATCH_METRICS_PATH = PRELIM_DIR / f'{MANIFEST_PATH.stem}_phase1_dataset_metrics.csv'
for path in [SLURM_SCRIPT_DIR, SLURM_PRINTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_TASK_SCRIPT = PROJECT_ROOT / 'scripts' / 'run_phase1_task.py'
SLURM_SCRIPT_PATH = SLURM_SCRIPT_DIR / f'{JOB_NAME}.slurm'

manifest_df = load_loci_manifest(MANIFEST_PATH)
validate_loci_manifest(manifest_df)
enabled_df = manifest_df[manifest_df['enabled']].copy().reset_index(drop=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Manifest path: {MANIFEST_PATH}')
print(f'Run mode: {RUN_MODE}')
print(f'Loci per task: {LOCI_PER_TASK}')
print(f'Enabled loci: {len(enabled_df)}')
display(enabled_df[['locus_id', 'gene_name', 'gene_id', 'gtex_tissue', 'gtex_chrom']].head(10))

In [ ]:
if len(enabled_df) == 0:
    raise ValueError(f'No enabled loci found in {MANIFEST_PATH}')

force_flag = ' --force' if FORCE_RERUN else ''
run_mode_flag = f' --run-mode {RUN_MODE}'
chunk_size_flag = f' --chunk-size {LOCI_PER_TASK}'
task_count = (len(enabled_df) + LOCI_PER_TASK - 1) // LOCI_PER_TASK

slurm_text = f"""#!/bin/bash
#SBATCH --job-name={JOB_NAME}
#SBATCH --array=1-{task_count}
#SBATCH --time={TIME_LIMIT}
#SBATCH --mem={MEMORY}
#SBATCH --cpus-per-task={CPUS_PER_TASK}
#SBATCH --mail-user={MAIL_USER}
#SBATCH --mail-type={MAIL_TYPE}
#SBATCH --output=/dev/null
#SBATCH --error=/dev/null
#SBATCH --account={ACCOUNT}

set -euo pipefail

PROJECT_ROOT=\"{PROJECT_ROOT}\"
MANIFEST_PATH=\"{MANIFEST_PATH}\"
RUN_TASK_SCRIPT=\"{RUN_TASK_SCRIPT}\"
SLURM_PRINTS_BASE=\"{SLURM_PRINTS_DIR}\"
BATCH_METRICS_PATH=\"{BATCH_METRICS_PATH}\"

JOBNAME=\"${{SLURM_JOB_NAME}}\"
PARENT_ID=\"${{SLURM_ARRAY_JOB_ID:-$SLURM_JOB_ID}}\"
TASK_ID=\"${{SLURM_ARRAY_TASK_ID:-0}}\"
PRINTS_DIR=\"${{SLURM_PRINTS_BASE}}/${{JOBNAME}}/${{PARENT_ID}}\"
mkdir -p \"${{PRINTS_DIR}}\"

exec >\"${{PRINTS_DIR}}/${{TASK_ID}}.out\" 2>\"${{PRINTS_DIR}}/${{TASK_ID}}.err\"
echo \"Batch metrics CSV: ${{BATCH_METRICS_PATH}}\"\n
echo \"[$(date -Is)] Starting task ${{TASK_ID}} for job ${{JOBNAME}} (parent ${{PARENT_ID}})\"\n

python \"${{RUN_TASK_SCRIPT}}\" --manifest \"${{MANIFEST_PATH}}\" --task-id \"${{TASK_ID}}\"{chunk_size_flag}{run_mode_flag}{force_flag}

echo \"[$(date -Is)] Completed task ${{TASK_ID}}\"\n
"""

SLURM_SCRIPT_PATH.write_text(slurm_text)
print(f'Wrote Slurm script: {SLURM_SCRIPT_PATH}')
print(slurm_text)